In [1]:
!pip install cdsapi -q
import os
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write("key:\n")
print('~/.cdsapirc file written')

~/.cdsapirc file written


## ERA5 data download

In [24]:
# Uses region-specific bounding boxes (CONUS, Alaska, Hawaii)
# Safe to re-run — skips already downloaded files
#
# Prerequisites:
#   1. CDS account: https://cds.climate.copernicus.eu
#   2. Accept T&C for: derived-era5-single-levels-daily-statistics
#      https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics
#   3. API key from your CDS profile page
#
# ============================================================

!pip install cdsapi xarray netcdf4 openpyxl tqdm matplotlib numpy -q

import os
import time
import numpy as np
import cdsapi
from getpass import getpass
from tqdm import tqdm
from datetime import datetime

# ============================================================
# API KEY SETUP — Secure input (won't show in notebook)
# ============================================================
print("=" * 60)
print("CDS API KEY SETUP")
print("=" * 60)

cdsapirc_path = os.path.expanduser('~/.cdsapirc')

# Check if already configured
if os.path.exists(cdsapirc_path):
    with open(cdsapirc_path, 'r') as f:
        content = f.read()
    print(f"\n  ✅ ~/.cdsapirc already exists")
    print(f"  URL: {[l for l in content.split(chr(10)) if 'url' in l][0] if 'url' in content else 'not found'}")
    print(f"  Key: ****{content.split('key:')[-1].strip()[-8:] if 'key' in content else 'not found'}")
    
    use_existing = input("\n  Use existing config? (y/n): ").strip().lower()
    if use_existing != 'y':
        api_key = getpass("  Enter your CDS API key: ")
        with open(cdsapirc_path, 'w') as f:
            f.write("url: https://cds.climate.copernicus.eu/api\n")
            f.write(f"key: {api_key}\n")
        print("  ✅ Updated ~/.cdsapirc")
else:
    print("\n  No ~/.cdsapirc found — setting up now")
    api_key = getpass("  Enter your CDS API key: ")
    with open(cdsapirc_path, 'w') as f:
        f.write("url: https://cds.climate.copernicus.eu/api\n")
        f.write(f"key: {api_key}\n")
    print("  ✅ Created ~/.cdsapirc")

# Test connection
print(f"\n  Testing CDS API connection...")
try:
    c = cdsapi.Client()
    print(f"  ✅ Connected to: {c.url}")
except Exception as e:
    print(f"  ❌ Connection failed: {e}")
    print(f"  Check your API key and internet connection")

In [25]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
ERA5_DIR = "../era5_monthly"

# Region bounding boxes [North, West, South, East]
# Hawaii expanded slightly to ensure good IDW comparison coverage
REGIONS = {
    "CONUS": {
        "area": [52, -128, 22, -64],
        "desc": "Conterminous US",
        "n_stations": 207,
    },
    "Alaska": {
        "area": [72, -170, 52, -130],
        "desc": "Alaska",
        "n_stations": 23,
    },
    "Hawaii": {
        "area": [23, -161, 18, -154],
        "desc": "Hawaiian Islands",
        "n_stations": 2,
        "note": "Matches Hawaii IDW gridded product extent",
    },
}

# ERA5 variables matched to USCRN
ERA5_VARIABLES = {
    "AirTemperature":      "2m_temperature",
    "DewpointTemperature": "2m_dewpoint_temperature",
    "Precipitation":       "total_precipitation",
    "SoilTemperature":     "soil_temperature_level_1",
    "SoilMoisture":        "volumetric_soil_water_layer_1",
}

YEARS = [str(y) for y in range(2006, 2022)]
MONTHS = [str(m).zfill(2) for m in range(1, 13)]

# Create directories
for region in REGIONS:
    os.makedirs(os.path.join(ERA5_DIR, region), exist_ok=True)

# Print plan
print(f"\n{'=' * 70}")
print("DOWNLOAD CONFIGURATION")
print(f"{'=' * 70}")
print(f"  Product:    reanalysis-era5-single-levels-monthly-means")
print(f"  Period:     2006-2021 (16 years × 12 months = 192 monthly values)")
print(f"  Variables:  {len(ERA5_VARIABLES)} ({', '.join(ERA5_VARIABLES.keys())})")
print(f"  Regions:    {len(REGIONS)} ({', '.join(REGIONS.keys())})")
print(f"  Total requests: {len(ERA5_VARIABLES) * len(REGIONS)} files")
print(f"  Estimated time: 1-3 hours\n")

print(f"  {'Region':<10} {'Stations':<10} {'Bbox [N,W,S,E]'}")
print(f"  {'─'*10} {'─'*10} {'─'*30}")
for r_name, r_info in REGIONS.items():
    print(f"  {r_name:<10} {r_info['n_stations']:<10} {r_info['area']}")

# ============================================================
# CELL 3: DOWNLOAD FUNCTION
# ============================================================
DELAY_BETWEEN = 5    # seconds between requests
RETRY_DELAY = 300    # 5 min wait on rate limit
MAX_RETRIES = 3

def download_era5_monthly(client, var_name, era5_var, region_name,
                          region_info, output_dir):
    """
    Download ERA5 monthly means — ONE file for entire 2006-2021 period.
    Returns dict with status info.
    """
    output_file = os.path.join(
        output_dir, region_name,
        f"ERA5_{var_name}_{region_name}_monthly_2006_2021.nc"
    )

    # Skip if exists and valid (>100KB)
    if os.path.exists(output_file) and os.path.getsize(output_file) > 100*1024:
        size = os.path.getsize(output_file) / (1024**2)
        return {
            "status": "skipped",
            "var": var_name,
            "region": region_name,
            "size_mb": round(size, 2),
        }

    # Remove incomplete files
    if os.path.exists(output_file):
        os.remove(output_file)

    # Try download with retries
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            time.sleep(DELAY_BETWEEN)
            start = time.time()

            client.retrieve(
                "reanalysis-era5-single-levels-monthly-means",
                {
                    "product_type": "monthly_averaged_reanalysis",
                    "variable": era5_var,
                    "year": YEARS,
                    "month": MONTHS,
                    "time": "00:00",
                    "area": region_info["area"],
                    "data_format": "netcdf",
                },
                output_file
            )

            elapsed = time.time() - start
            size = os.path.getsize(output_file) / (1024**2)

            return {
                "status": "success",
                "var": var_name,
                "region": region_name,
                "size_mb": round(size, 2),
                "time_min": round(elapsed/60, 1),
                "attempt": attempt,
            }

        except Exception as e:
            error = str(e)

            if "403" in error or "rate" in error.lower():
                if attempt < MAX_RETRIES:
                    wait = RETRY_DELAY * attempt
                    print(f"\n      ⚠️ Rate limited — waiting {wait//60} min "
                          f"(attempt {attempt}/{MAX_RETRIES})")
                    time.sleep(wait)
                    try:
                        client = cdsapi.Client(quiet=True)
                    except:
                        pass
                else:
                    return {"status": "failed", "var": var_name,
                            "region": region_name, "error": error[:100]}
            elif attempt < MAX_RETRIES:
                print(f"\n      ⚠️ Error — retrying in 60s "
                      f"(attempt {attempt}/{MAX_RETRIES}): {error[:60]}")
                time.sleep(60)
            else:
                return {"status": "failed", "var": var_name,
                        "region": region_name, "error": error[:100]}

    return {"status": "failed", "var": var_name,
            "region": region_name, "error": "Max retries exceeded"}

print("✅ Download function defined")
print(f"   Delay: {DELAY_BETWEEN}s | Retries: {MAX_RETRIES} | "
      f"Backoff: {RETRY_DELAY//60} min")

# ============================================================
# CELL 4: PRE-DOWNLOAD CHECK
# ============================================================
print(f"\n{'=' * 70}")
print("PRE-DOWNLOAD STATUS CHECK")
print(f"{'=' * 70}")

total_needed = 0
total_done = 0
total_size_mb = 0

print(f"\n  {'Region':<10} {'Variable':<22} {'Status':<15} {'Size'}")
print(f"  {'─'*10} {'─'*22} {'─'*15} {'─'*10}")

for region_name in REGIONS:
    for var_name in ERA5_VARIABLES:
        fp = os.path.join(
            ERA5_DIR, region_name,
            f"ERA5_{var_name}_{region_name}_monthly_2006_2021.nc"
        )
        if os.path.exists(fp) and os.path.getsize(fp) > 100*1024:
            size = os.path.getsize(fp) / (1024**2)
            total_size_mb += size
            total_done += 1
            status = "✅ Done"
            size_str = f"{size:.1f} MB"
        else:
            total_needed += 1
            status = "⏳ Needed"
            size_str = "—"
        print(f"  {region_name:<10} {var_name:<22} {status:<15} {size_str}")

print(f"\n  Already downloaded: {total_done}")
print(f"  Need to download:   {total_needed}")
print(f"  Existing size:      {total_size_mb:.1f} MB")

if total_needed > 0:
    est_time = total_needed * 5  # ~5 min per file (queue wait)
    print(f"  ⏱️ Estimated time:  ~{est_time} min ({est_time/60:.1f} hours)")

# ============================================================
# CELL 5: DOWNLOAD ALL FILES
# ============================================================
if total_needed > 0:
    print(f"\n{'=' * 70}")
    print("DOWNLOADING ERA5 MONTHLY DATA")
    print(f"{'=' * 70}\n")

    c = cdsapi.Client(quiet=True)
    download_results = []
    start_time = time.time()

    for region_name, region_info in REGIONS.items():
        print(f"\n  📍 {region_name} ({region_info['desc']})")
        print(f"     Area: {region_info['area']}")

        for var_name, era5_var in ERA5_VARIABLES.items():
            print(f"\n     📥 {var_name}...", end=" ", flush=True)

            result = download_era5_monthly(
                c, var_name, era5_var,
                region_name, region_info, ERA5_DIR
            )
            download_results.append(result)

            if result['status'] == 'success':
                print(f"✅ {result['size_mb']} MB ({result.get('time_min', 0)} min)")
            elif result['status'] == 'skipped':
                print(f"⏭ already done ({result['size_mb']} MB)")
            else:
                print(f"❌ {result.get('error', 'unknown')[:60]}")

    elapsed_total = (time.time() - start_time) / 60

    # Summary
    n_success = sum(1 for r in download_results if r['status'] == 'success')
    n_skipped = sum(1 for r in download_results if r['status'] == 'skipped')
    n_failed = sum(1 for r in download_results if r['status'] == 'failed')

    print(f"\n{'=' * 70}")
    print(f"DOWNLOAD SUMMARY")
    print(f"{'=' * 70}")
    print(f"  Total time:    {elapsed_total:.1f} minutes")
    print(f"  ✅ Success:    {n_success}")
    print(f"  ⏭ Skipped:    {n_skipped}")
    print(f"  ❌ Failed:     {n_failed}")

    if n_failed > 0:
        print(f"\n  Failed downloads:")
        for r in download_results:
            if r['status'] == 'failed':
                print(f"    ❌ {r['region']}/{r['var']}: "
                      f"{r.get('error', '')[:60]}")
        print(f"\n  💡 Re-run this cell to retry failed downloads")
else:
    print(f"\n  ✅ All files already downloaded!")

## Compute Relative Humidity from Dew Point Temperature Data

In [26]:
import os
import xarray as xr   # ← This was missing!
import numpy as np
from datetime import datetime

ERA5_DIR = "../era5_monthly"

REGIONS = ["CONUS", "Alaska", "Hawaii"]

# ============================================================
# Compute Relative Humidity (Magnus Formula)
# ============================================================
print("=" * 70)
print("COMPUTING RELATIVE HUMIDITY FROM T2m + Td2m")
print("=" * 70)
print("  Magnus formula:")
print("  RH = 100 × exp(17.625·Td / (243.04+Td)) / exp(17.625·T / (243.04+T))\n")

for region_name in REGIONS:
    region_dir = os.path.join(ERA5_DIR, region_name)
    
    t2m_file = os.path.join(region_dir,
        f"ERA5_AirTemperature_{region_name}_monthly_2006_2021.nc")
    d2m_file = os.path.join(region_dir,
        f"ERA5_DewpointTemperature_{region_name}_monthly_2006_2021.nc")
    rh_file = os.path.join(region_dir,
        f"ERA5_RelativeHumidity_{region_name}_monthly_2006_2021.nc")
    
    if os.path.exists(rh_file) and os.path.getsize(rh_file) > 50*1024:
        size = os.path.getsize(rh_file) / (1024**2)
        print(f"  ⏭ {region_name}: RH already computed ({size:.1f} MB)")
        continue
    
    if not os.path.exists(t2m_file):
        print(f"  ❌ {region_name}: AirTemperature file missing")
        continue
    if not os.path.exists(d2m_file):
        print(f"  ❌ {region_name}: DewpointTemperature file missing")
        continue
    
    print(f"  Computing RH for {region_name}...", end=" ", flush=True)
    
    try:
        t2m_ds = xr.open_dataset(t2m_file)
        d2m_ds = xr.open_dataset(d2m_file)
        
        # Find variable names (skip coordinate-like names)
        skip_vars = {'time', 'valid_time', 'latitude', 'longitude',
                     'lat', 'lon', 'expver', 'number'}
        t2m_var = [v for v in t2m_ds.data_vars
                   if v.lower() not in skip_vars][0]
        d2m_var = [v for v in d2m_ds.data_vars
                   if v.lower() not in skip_vars][0]
        
        # Convert K → °C
        T = t2m_ds[t2m_var] - 273.15
        Td = d2m_ds[d2m_var] - 273.15
        
        # Magnus formula
        RH = 100.0 * (
            np.exp((17.625 * Td) / (243.04 + Td)) /
            np.exp((17.625 * T) / (243.04 + T))
        )
        RH = RH.clip(0, 100)
        
        # Save
        rh_ds = RH.to_dataset(name="relative_humidity")
        rh_ds["relative_humidity"].attrs = {
            "units": "%",
            "long_name": "Relative Humidity",
            "source": "Computed from ERA5 2m_temperature and 2m_dewpoint_temperature",
            "method": "Magnus formula",
        }
        rh_ds.attrs = {
            "title": f"ERA5 Relative Humidity (computed) — {region_name}",
            "region": region_name,
            "source": "Magnus formula from ERA5 T2m + Td2m",
            "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
        rh_ds.to_netcdf(rh_file)
        
        size = os.path.getsize(rh_file) / (1024**2)
        rh_min = float(RH.min())
        rh_max = float(RH.max())
        rh_mean = float(RH.mean())
        print(f"✅ ({size:.1f} MB | range: {rh_min:.1f}-{rh_max:.1f}%, mean: {rh_mean:.1f}%)")
        
        t2m_ds.close()
        d2m_ds.close()
    except Exception as e:
        print(f"❌ {e}")

# ============================================================
# Verify ALL Files (including newly computed RH)
# ============================================================
print(f"\n{'=' * 70}")
print("VERIFICATION & FILE SUMMARY")
print(f"{'=' * 70}")

ERA5_VARS = ["AirTemperature", "DewpointTemperature", "Precipitation",
             "SoilTemperature", "SoilMoisture", "RelativeHumidity"]
total_size_gb = 0

print(f"\n  {'Variable':<25} {'CONUS':<14} {'Alaska':<14} {'Hawaii'}")
print(f"  {'─'*25} {'─'*14} {'─'*14} {'─'*14}")

for var_name in ERA5_VARS:
    row = f"  {var_name:<25}"
    for region_name in REGIONS:
        fp = os.path.join(
            ERA5_DIR, region_name,
            f"ERA5_{var_name}_{region_name}_monthly_2006_2021.nc"
        )
        if os.path.exists(fp) and os.path.getsize(fp) > 50*1024:
            s = os.path.getsize(fp) / (1024**2)
            total_size_gb += s
            row += f"{'✅ ' + f'{s:.1f}MB':<14}"
        else:
            row += f"{'❌ missing':<14}"
    print(row)

total_size_gb = total_size_gb / 1024
print(f"\n  Total ERA5 monthly data: {total_size_gb*1024:.0f} MB ({total_size_gb:.2f} GB)")

# ============================================================
# Data Inspection
# ============================================================
print(f"\n{'=' * 70}")
print("DATA INSPECTION (sample: AirTemperature per region)")
print(f"{'=' * 70}")

skip_vars = {'time','valid_time','latitude','longitude','lat','lon','expver','number'}

for region_name in REGIONS:
    fp = os.path.join(
        ERA5_DIR, region_name,
        f"ERA5_AirTemperature_{region_name}_monthly_2006_2021.nc"
    )
    if not os.path.exists(fp):
        print(f"\n  ❌ {region_name}: file not found")
        continue
    
    try:
        ds = xr.open_dataset(fp)
        var = [v for v in ds.data_vars if v.lower() not in skip_vars][0]
        time_coord = 'time' if 'time' in ds.coords else 'valid_time'
        lat_coord = 'latitude' if 'latitude' in ds.coords else 'lat'
        lon_coord = 'longitude' if 'longitude' in ds.coords else 'lon'
        
        print(f"\n  📍 {region_name}")
        print(f"     Variable: {var}")
        print(f"     Dims:     {dict(ds.dims)}")
        print(f"     Time:     {len(ds[time_coord])} months "
              f"({str(ds[time_coord].values[0])[:10]} → "
              f"{str(ds[time_coord].values[-1])[:10]})")
        print(f"     Spatial:  {len(ds[lat_coord])} × {len(ds[lon_coord])} @ 0.25°")
        print(f"     Lat:      {float(ds[lat_coord].min()):.2f} → {float(ds[lat_coord].max()):.2f}")
        print(f"     Lon:      {float(ds[lon_coord].min()):.2f} → {float(ds[lon_coord].max()):.2f}")
        ds.close()
    except Exception as e:
        print(f"\n  ❌ {region_name}: {e}")

print(f"""
{'=' * 70}
✅ ERA5 DOWNLOAD COMPLETE
{'=' * 70}

  All 18 files ready (5 downloaded variables + RH × 3 regions)

  📁 Files in: {ERA5_DIR}/
""")

### ERA5 Precipitation CONVERSION 

In [ ]:
import numpy as np

def era5_precip_correct(x):
    """ERA5 'tp' avgad is m/day. Convert to mm/day: × 1000 only."""
    return x * 1000.0

old_fn = VARIABLE_CONFIG['Precipitation']['era5_convert']
VARIABLE_CONFIG['Precipitation']['era5_convert'] = era5_precip_correct

print("✅ Patched VARIABLE_CONFIG['Precipitation']['era5_convert']")
print(f"   OLD: lambda x: x * 1000 * 30  (assumed monthly total)")
print(f"   NEW: lambda x: x * 1000        (correct: m/day → mm/day)")

# Verify
test = np.array([0.00236])  # typical ERA5 value
print(f"\n   Test: ERA5 input 0.00236 m/day → {era5_precip_correct(test)[0]:.3f} mm/day")
print(f"   ✅ Correct ERA5 magnitude restored")

### ERA5 DATA AUDIT

In [27]:
# ============================================================
# ERA5 DATA AUDIT — Check structure, coverage, and units
# ============================================================
import os
import numpy as np
import pandas as pd
import xarray as xr
import glob

print("=" * 70)
print("ERA5 DATA AUDIT")
print("=" * 70)
ERA5_DIR="../era5_monthly/"
# ── Find all ERA5 files ──
all_era5 = sorted(glob.glob(os.path.join(ERA5_DIR, "**", "*.nc"), recursive=True))
print(f"\n  📁 Found {len(all_era5)} ERA5 NetCDF files in {ERA5_DIR}")

if not all_era5:
    raise FileNotFoundError(f"No .nc files found under {ERA5_DIR}")

# Group by region
files_by_region = {'CONUS': [], 'Alaska': [], 'Hawaii': [], 'Other': []}
for f in all_era5:
    rel = os.path.relpath(f, ERA5_DIR)
    placed = False
    for region in ['CONUS', 'Alaska', 'Hawaii']:
        if region.lower() in rel.lower():
            files_by_region[region].append(f)
            placed = True
            break
    if not placed:
        files_by_region['Other'].append(f)

print(f"\n  Files per region:")
for r, fs in files_by_region.items():
    print(f"     {r:<10} {len(fs)} files")

# ── Detailed audit for each file ──
audit = []
for region in ['CONUS', 'Alaska', 'Hawaii']:
    if not files_by_region[region]:
        continue
    
    print(f"\n{'█' * 70}")
    print(f"  📍 REGION: {region}")
    print(f"{'█' * 70}")
    
    for fp in files_by_region[region]:
        fname = os.path.basename(fp)
        print(f"\n  📄 {fname}")
        
        try:
            ds = xr.open_dataset(fp)
            
            # Identify data variable
            skip = {'time','valid_time','latitude','longitude','lat','lon','expver','number'}
            data_vars = [v for v in ds.data_vars if v.lower() not in skip]
            
            # Identify time coord
            tc = 'valid_time' if 'valid_time' in ds.coords else 'time'
            lat_c = 'latitude' if 'latitude' in ds.coords else 'lat'
            lon_c = 'longitude' if 'longitude' in ds.coords else 'lon'
            
            # Time info
            times = pd.to_datetime(ds[tc].values)
            
            # Lat/lon info
            lats = ds[lat_c].values
            lons = ds[lon_c].values
            
            # Each data variable
            for dv in data_vars:
                arr = ds[dv]
                if 'expver' in arr.dims: arr = arr.isel(expver=0)
                if 'number' in arr.dims: arr = arr.isel(number=0)
                
                vals = arr.values
                vals_flat = vals.flatten()
                vals_valid = vals_flat[~np.isnan(vals_flat)]
                
                attrs = arr.attrs
                units = attrs.get('units', 'no units attr')
                long_name = attrs.get('long_name', dv)
                
                row = {
                    "Region": region,
                    "File": fname,
                    "Variable": dv,
                    "Long_Name": long_name,
                    "Units": units,
                    "Shape": str(arr.shape),
                    "N_times": len(times),
                    "Time_start": times.min().strftime('%Y-%m-%d') if len(times) > 0 else 'N/A',
                    "Time_end": times.max().strftime('%Y-%m-%d') if len(times) > 0 else 'N/A',
                    "Lat_min": round(float(lats.min()), 2),
                    "Lat_max": round(float(lats.max()), 2),
                    "Lat_step": round(float(abs(lats[1]-lats[0])), 4) if len(lats) > 1 else 'N/A',
                    "Lon_min": round(float(lons.min()), 2),
                    "Lon_max": round(float(lons.max()), 2),
                    "Lon_step": round(float(abs(lons[1]-lons[0])), 4) if len(lons) > 1 else 'N/A',
                    "Valid_pct": round(100*len(vals_valid)/vals_flat.size, 1) if vals_flat.size > 0 else 0,
                    "Min": round(float(vals_valid.min()), 3) if len(vals_valid) > 0 else 'N/A',
                    "Max": round(float(vals_valid.max()), 3) if len(vals_valid) > 0 else 'N/A',
                    "Mean": round(float(vals_valid.mean()), 3) if len(vals_valid) > 0 else 'N/A',
                }
                audit.append(row)
                
                print(f"     Variable:    {dv} ({long_name})")
                print(f"     Units:       {units}")
                print(f"     Shape:       {arr.shape}")
                print(f"     Time range:  {times.min().strftime('%Y-%m-%d')} → "
                      f"{times.max().strftime('%Y-%m-%d')} ({len(times)} steps)")
                print(f"     Lat:         {lats.min():.2f}° to {lats.max():.2f}° "
                      f"(step {abs(lats[1]-lats[0]):.4f}°)")
                print(f"     Lon:         {lons.min():.2f}° to {lons.max():.2f}° "
                      f"(step {abs(lons[1]-lons[0]):.4f}°)")
                print(f"     Valid data:  {100*len(vals_valid)/vals_flat.size:.1f}%")
                if len(vals_valid) > 0:
                    print(f"     Range:       [{vals_valid.min():.3f}, {vals_valid.max():.3f}]  "
                          f"mean={vals_valid.mean():.3f}")
            
            ds.close()
        except Exception as e:
            print(f"     ❌ ERROR: {e}")

# ── Save audit table ──
audit_df = pd.DataFrame(audit)
audit_csv = os.path.join(STATS_DIR, "era5_audit.csv")
audit_df.to_csv(audit_csv, index=False)

# ── Coverage matrix ──
print(f"\n{'=' * 70}")
print("COVERAGE MATRIX: which (region × variable) are available?")
print(f"{'=' * 70}\n")

expected_vars = list(VARIABLE_CONFIG.keys())
print(f"  {'Variable':<22} {'CONUS':<10} {'Alaska':<10} {'Hawaii':<10}")
print(f"  {'─'*22} {'─'*10} {'─'*10} {'─'*10}")

coverage_issues = []
for var_name in expected_vars:
    row_str = f"  {var_name:<22}"
    for region in ['CONUS', 'Alaska', 'Hawaii']:
        # Look for file matching this region+variable
        expected_fn = f"ERA5_{var_name}_{region}_monthly_2006_2021.nc"
        full_path = os.path.join(ERA5_DIR, region, expected_fn)
        
        if os.path.exists(full_path):
            sz_mb = os.path.getsize(full_path) / (1024**2)
            row_str += f" ✅ {sz_mb:>5.1f}MB"
        else:
            row_str += f" ❌ MISSING"
            coverage_issues.append(f"{region}/{var_name}")
    print(row_str)

# ── Unit check (catches conversion bugs) ──
print(f"\n{'=' * 70}")
print("UNIT CHECK: comparing ERA5 units vs expected after conversion")
print(f"{'=' * 70}\n")

EXPECTED_RANGES = {
    'AirTemperature':   {'units': '°C',     'plausible': (-60, 50)},
    'SoilTemperature':  {'units': '°C',     'plausible': (-50, 45)},
    'Precipitation':    {'units': 'mm',     'plausible': (0, 500)},
    'RelativeHumidity': {'units': '%',      'plausible': (0, 100)},
    'SoilMoisture':     {'units': 'm³/m³',  'plausible': (0, 1.0)},
}

print(f"  {'Variable':<22} {'Region':<10} {'Native units':<15} {'Native range':<22} "
      f"{'After conversion':<22} {'Status'}")
print(f"  {'─'*22} {'─'*10} {'─'*15} {'─'*22} {'─'*22} {'─'*8}")

for _, row in audit_df.iterrows():
    var_name = None
    for vn in VARIABLE_CONFIG:
        if vn in row['File']:
            var_name = vn
            break
    if var_name is None:
        continue
    
    var_cfg = VARIABLE_CONFIG[var_name]
    expected = EXPECTED_RANGES.get(var_name, {})
    
    # Apply conversion to test min/max
    try:
        nv_min = float(row['Min'])
        nv_max = float(row['Max'])
        cv_min = float(var_cfg['era5_convert'](np.array([nv_min]))[0])
        cv_max = float(var_cfg['era5_convert'](np.array([nv_max]))[0])
        
        plaus = expected.get('plausible', (None, None))
        if plaus[0] is not None:
            ok = plaus[0] <= cv_min <= plaus[1] and plaus[0] <= cv_max <= plaus[1]
            status = "✅" if ok else "⚠️ OUT OF RANGE"
        else:
            status = "?"
        
        print(f"  {var_name:<22} {row['Region']:<10} {row['Units']:<15} "
              f"[{nv_min:>8.2f}, {nv_max:>8.2f}]  "
              f"[{cv_min:>8.2f}, {cv_max:>8.2f}]  {status}")
    except Exception as e:
        print(f"  {var_name:<22} {row['Region']:<10} ERROR: {e}")

# ── Final summary ──
print(f"\n{'=' * 70}")
print("ERA5 AUDIT SUMMARY")
print(f"{'=' * 70}")

print(f"\n  📊 Total files:        {len(all_era5)}")
print(f"  📊 Variables found:    {audit_df['Variable'].nunique() if len(audit_df)>0 else 0}")
print(f"  📊 Regions covered:    {audit_df['Region'].nunique() if len(audit_df)>0 else 0}")
print(f"  📊 Audit CSV saved to: {audit_csv}")

if coverage_issues:
    print(f"\n  ⚠️ Missing combinations ({len(coverage_issues)}):")
    for c in coverage_issues:
        print(f"     • {c}")
else:
    print(f"\n  ✅ All expected (region × variable) combinations present")

print(f"\n  🎯 Ready to use ERA5 for validation")